# Task 1.3: What the Paper Claims to Improve

**Paper:** Exact Discovery of Time Series Motifs  
**Authors:** Abdullah Mueen, Eamonn Keogh, Qiang Zhu, Sydney Cash, Brandon Westover  
**Venue:** KDD 2009 (ACM SIGKDD)  
**Student:** Abhishek (m23csai230137)

---

## Main Baseline / Prior Method

The paper's primary baseline is the **Brute Force Motif Discovery** algorithm (Table 1, Section 3.2). This is a simple nested loop that compares every possible pair of time series in the database and returns the pair with the smallest Euclidean distance. The paper also compares against **Brute Force with Early Abandoning**, which is the same nested loop but stops each individual distance computation early if the cumulative squared error exceeds the current best-so-far (Figure 2, Section 2).

Additionally, the paper benchmarks against numerous **approximate motif discovery algorithms** from the literature, including methods by Chiu et al. [7], Meng et al. [23], Beaudoin et al. [4], and the FLAME algorithm [33]. However, the paper dismisses all of them since their exact algorithm is faster even than these approximate methods (Section 1.1).

## Limitation of the Baseline Identified by the Paper

The fundamental limitation of brute force search is its **O(m²n) time complexity**. For a database of m time series each of length n, brute force must compute the full Euclidean distance for all m(m−1)/2 pairs, and each distance computation costs O(n). The paper reports that for 100,000 random walk time series of length 1024, brute force takes **12.7 hours** (Section 4.1, Figure 6). Even with early abandoning optimisation, brute force still requires examining every pair — it only saves time within each individual distance computation, not in reducing the number of pairs checked. This makes exact motif discovery on large real-world datasets (like the 180,000-subsequence EEG dataset in Section 5.3) effectively intractable.

## How the Proposed Method Overcomes This Limitation

The MK algorithm overcomes this by **pruning the vast majority of candidate pairs without ever computing their true distance**. It uses pre-computed distances to multiple random reference points and the triangle inequality to derive cheap lower bounds. Pairs whose lower bounds exceed the current best-so-far are skipped entirely. Combined with a smart offset-based search ordering (which checks nearby pairs in the 1D projection first) and a termination condition (Lemma 1 — once an entire offset level produces no surviving candidates, all higher offsets can be skipped), the algorithm typically examines only a tiny fraction of all O(m²) pairs before provably finding the exact motif.

## Condition Where MK Would NOT Outperform the Baseline

The MK algorithm would **not** outperform brute force in a scenario where all pairwise distances in the database are nearly identical — i.e., every pair of time series is approximately the same distance apart, and no pair is significantly closer than the rest. This could happen with a small database of **independent white noise** sequences of the same length and variance.

In this scenario:
1. **Triangle inequality lower bounds would be near zero:** Since all time series would be roughly equidistant from any reference point, `|d(ref, Dᵢ) - d(ref, Dⱼ)|` would be approximately zero for all pairs, providing no pruning power.
2. **The best-so-far would remain large:** Without a clearly close pair, the best-so-far would stay near the average distance, meaning early abandoning would rarely trigger.
3. **Overhead without benefit:** The MK algorithm would still incur the extra cost of computing R × m reference distances and sorting, but would fail to prune any pairs, making it strictly **slower** than plain brute force due to this overhead.

The paper itself acknowledges this: *"Clearly this strategy has the worst case complexity O(m²), which is equal to the brute force algorithm. However this only occurs in the cases where the motif has distance larger than any lower bound computed using a random reference"* (Section 3.2.1). The paper also notes that *"Random walk data is a difficult case for our algorithm, since we should not expect a very close motif pair to exist"* (Section 4.1), confirming that the speedup is smallest when no tight motif exists.